# 01 — Paper discovery and metadata

Owner: Member 1. Finds every paper in the assigned ID range, works out how many versions
each one has, and stores per-version metadata in `<paper-folder>/metadata.json`.

Functions provided to the rest of the pipeline (agreed interface):

| Function | Returns |
|---|---|
| `list_ids(start, end)` | `list[str]` of arXiv IDs in the range (inclusive) |
| `all_assigned_ids()` | `list[str]` of every ID in `ID_RANGES` (20,000 for our group) |
| `fetch_metadata(ids)` | `list[dict]`, one per requested ID: `id, status, versions, title, authors, submitted, revised, venue, version_info` |
| `save_metadata(paper)` | writes `metadata.json`, returns its path (`None` if the paper was not found) |
| `load_metadata(arxiv_id)` | the saved dict, or `None` |
| `discover_all()` | `(papers, queried, seconds)`: every assigned ID, resumed from `MANIFEST` and the batch checkpoints |

`status` is `ok` (all versions found), `partial` (some older version could not be fetched),
`not_found` (ID does not exist on arXiv) or `error` (the request kept failing).

### How it works

1. **Pass 1, latest versions.** Query the arXiv API with up to 100 plain IDs per call
   (`id_list`). arXiv returns the *latest* version of each paper, e.g. `1706.03762v7`, so the
   suffix gives the number of versions. IDs that do not exist are simply missing from the reply.
2. **Pass 2, older versions.** Query the versioned IDs `v1 … v(n-1)`, again 100 per call.
   For a versioned ID the API returns that version's own title, authors, abstract and date
   (`updated`), which gives the revision dates and per-version metadata.
3. Batching keeps the number of calls low: with the mandatory 3 s pause, 1,000 papers need
   about 10 calls in pass 1 instead of 1,000.
4. **Checkpoints.** `discover_all` saves every finished batch of 100 IDs to `DISCOVERY_BATCHES`
   and the full result to `MANIFEST`, so a restart only queries what is still missing.

**Error handling.**
- HTTP 429 (too many requests): wait `ARXIV_BACKOFF` seconds (doubling each time) and retry the same batch.
- Any other failure (e.g. HTTP 500): split the batch in half and retry each half, down to single
  IDs; an ID that still fails is reported with status `error`.
- The API always answers HTTP 500 for *versioned old-style* IDs (`hep-th/9711200v1`), so pass 2
  skips them: old-style papers get the latest version's metadata and empty records for older
  versions (status `partial`). All our assigned ranges are new-style, so this does not affect the crawl.

Library: `arxiv.py` 4.x, used only through `Client.results(Search(...))` as the lab book requires.

In [ ]:
if "CONFIG_LOADED" not in globals():
    get_ipython().run_line_magic("run", "-i ./00_config.ipynb")

import hashlib

import arxiv
import requests
try:
    from tqdm.auto import tqdm   # progress bars (widgets in Jupyter / Colab, text elsewhere)
except ImportError:
    tqdm = None                  # no bars: plain progress lines instead
try:
    # Colab draws ipywidgets 8 (the tqdm bars) only with its custom widget manager; without it the bars
    # vanish and Colab just suggests the enable_custom_widget_manager snippet. Lasts for the session.
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
except ImportError:
    pass                         # not on Colab

## 1. Build the ID list

In [ ]:
def list_ids(start, end):
    """All arXiv IDs from start to end (inclusive), e.g. list_ids('2310.00001', '2310.00003').

    Works for new-style IDs (same YYMM) and old-style IDs (same archive/YYMM),
    keeping the zero-padding of the input.
    """
    new_a, new_b = NEW_ID_RE.match(start), NEW_ID_RE.match(end)
    old_a, old_b = OLD_ID_RE.match(start), OLD_ID_RE.match(end)

    if new_a and new_b:
        if new_a.group(1) != new_b.group(1):
            raise ValueError("start and end must have the same YYMM prefix")
        prefix = new_a.group(1) + "."
        width = len(new_a.group(2))
        lo, hi = int(new_a.group(2)), int(new_b.group(2))
    elif old_a and old_b:
        if old_a.group(1) != old_b.group(1) or old_a.group(2)[:4] != old_b.group(2)[:4]:
            raise ValueError("start and end must have the same archive/YYMM prefix")
        prefix = f"{old_a.group(1)}/{old_a.group(2)[:4]}"
        width = 3
        lo, hi = int(old_a.group(2)[4:]), int(old_b.group(2)[4:])
    else:
        raise ValueError(f"unrecognised ID format: {start!r}, {end!r}")

    if lo > hi:
        raise ValueError("start must not be after end")
    return [f"{prefix}{n:0{width}d}" for n in range(lo, hi + 1)]


def all_assigned_ids(ranges=None):
    """Every ID in the group's assigned ranges (ID_RANGES by default), without duplicates."""
    ranges = ID_RANGES if ranges is None else ranges
    ids = [i for start, end in ranges for i in list_ids(start, end)]
    return list(dict.fromkeys(ids))

## 2. Query the arXiv API

One shared `Client` keeps the 3-second pause between *all* discovery requests.
`DISCOVERY_STATS` collects numbers for the report (calls made, IDs that failed, time spent).

In [ ]:
ARXIV_CLIENT = arxiv.Client(
    page_size=ARXIV_BATCH_SIZE,
    delay_seconds=ARXIV_DELAY,
    num_retries=1,   # quick retry only; 429 and bad batches are handled in _query_arxiv
)

DISCOVERY_STATS = {"queries": 0, "failed_ids": [], "seconds": 0.0}


def _chunks(items, size):
    for i in range(0, len(items), size):
        yield items[i:i + size]


def _split_or_give_up(id_list, err):
    """Retry each half of a failing batch, so one bad ID cannot block the other 99."""
    if len(id_list) == 1:
        print(f"[discovery] giving up on {id_list[0]}: {err}")
        DISCOVERY_STATS["failed_ids"].append(id_list[0])
        return []
    mid = len(id_list) // 2
    return _query_arxiv(id_list[:mid]) + _query_arxiv(id_list[mid:])


def _query_arxiv(id_list, _attempt=0):
    """arxiv.Result objects for id_list; IDs that do not exist are absent from the result.

    HTTP 429 (too many requests): wait ARXIV_BACKOFF, doubling each time, then retry the same batch.
    Splitting would only send more requests, so after ARXIV_RETRIES the whole batch is marked failed.
    Other errors (e.g. HTTP 500): split the batch in half and retry each half.
    """
    DISCOVERY_STATS["queries"] += 1
    try:
        search = arxiv.Search(id_list=id_list, max_results=len(id_list))
        return list(ARXIV_CLIENT.results(search))
    except arxiv.HTTPError as err:
        if err.status != 429:
            return _split_or_give_up(id_list, err)
        if _attempt < ARXIV_RETRIES:
            wait = ARXIV_BACKOFF * 2 ** _attempt
            print(f"[discovery] HTTP 429, waiting {wait:.0f} s before retrying")
            time.sleep(wait)
            return _query_arxiv(id_list, _attempt + 1)
        print(f"[discovery] still rate-limited, giving up on {len(id_list)} IDs (rerun later)")
        DISCOVERY_STATS["failed_ids"].extend(id_list)
        return []
    except (arxiv.UnexpectedEmptyPageError, requests.RequestException) as err:
        return _split_or_give_up(id_list, err)

## 3. Turn API results into metadata records

In [ ]:
def _clean(text):
    """Collapse the line breaks and double spaces arXiv leaves in titles and abstracts."""
    return " ".join(text.split()) if text else text


def _version_record(result):
    """Metadata of one version, taken from the API entry for that version."""
    _, version = split_version(result.get_short_id())
    return {
        "version": version,
        "date": to_iso(result.updated),   # submission date of this version
        "title": _clean(result.title),
        "authors": [a.name for a in result.authors],
        "abstract": _clean(result.summary),
        "journal_ref": result.journal_ref,
        "doi": result.doi,
        "comment": result.comment,
        "categories": result.categories,
    }


def fetch_metadata(ids):
    """Metadata of every version of every paper in ids (one dict per distinct input ID)."""
    start = time.perf_counter()
    # Version suffixes in the input are ignored: we always want all versions
    ids = list(dict.fromkeys(split_version(i)[0] for i in ids))

    # Pass 1: latest version of each paper -> number of versions
    latest = {}
    for batch in _chunks(ids, ARXIV_BATCH_SIZE):
        for result in _query_arxiv(batch):
            base, n_versions = split_version(result.get_short_id())
            latest[base] = (n_versions, result)

    # Pass 2: every older version, by its versioned ID.
    # The API answers HTTP 500 for versioned old-style IDs (hep-th/9711200v1), so those are
    # skipped and keep empty records. Our assigned ranges contain only new-style IDs.
    wanted = [f"{base}v{v}" for base, (n, _) in latest.items()
              if NEW_ID_RE.match(base) for v in range(1, n)]
    older = {}
    for batch in _chunks(wanted, ARXIV_BATCH_SIZE):
        for result in _query_arxiv(batch):
            older[result.get_short_id()] = result

    failed = set(DISCOVERY_STATS["failed_ids"])
    papers = []
    for base in ids:
        if base not in latest:
            papers.append({"id": base, "status": "error" if base in failed else "not_found",
                           "versions": 0})
            continue

        n_versions, newest = latest[base]
        version_info = []
        for v in range(1, n_versions + 1):
            result = newest if v == n_versions else older.get(f"{base}v{v}")
            # A version whose query failed keeps an empty record instead of being dropped
            version_info.append(_version_record(result) if result else {"version": v, "date": None})

        papers.append({
            "id": base,
            "status": "ok" if all(r["date"] for r in version_info) else "partial",
            "versions": n_versions,
            "title": _clean(newest.title),
            "authors": [a.name for a in newest.authors],
            "submitted": to_iso(newest.published),               # date of v1
            "revised": [r["date"] for r in version_info[1:]],    # dates of v2..vN
            "venue": newest.journal_ref,                         # None if not published
            "version_info": version_info,
            "fetched_at": to_iso(datetime.now(timezone.utc)),
        })

    DISCOVERY_STATS["seconds"] += time.perf_counter() - start
    return papers

## 4. Save and load `metadata.json`

In [ ]:
def save_metadata(paper):
    """Write <paper-folder>/metadata.json. Papers that do not exist on arXiv get no folder."""
    if paper["status"] in ("not_found", "error"):
        return None
    path = paper_dir(paper["id"]) / "metadata.json"
    write_json(path, paper)
    return path


def load_metadata(arxiv_id):
    return read_json(DATA_DIR / to_folder_id(arxiv_id) / "metadata.json")

## 5. Discover every assigned ID, with checkpoints

`resolve_discovery_batches` queries up to 100 IDs per batch and saves each completed batch to
`DISCOVERY_BATCHES` before the next API call. A batch that still fails after retries stops the run;
a later run reuses the completed batches.

`discover_all` is what `05_run_pipeline.ipynb` calls. It loads `MANIFEST` if it exists, re-queries only
the IDs that are missing or failed last time (`status == "error"`), and saves the merged manifest.
`discovery_seconds` adds up the API time of all runs, for the report.

In [ ]:
def resolve_discovery_batches(ids, fetch, save, batch_dir, failed_ids, batch_size=100):
    """Discover in small batches; persist completed batches before the next API call."""
    ids = list(ids)
    if batch_size < 1 or len(ids) != len(set(ids)):
        raise ValueError("Invalid discovery batch size or duplicate IDs")
    batch_dir = Path(batch_dir)
    batch_dir.mkdir(parents=True, exist_ok=True)
    records, seconds, queried = [], 0.0, False
    bar = tqdm(total=len(ids), unit="ID", desc="Discovery", dynamic_ncols=True) if tqdm else None
    try:
        for offset in range(0, len(ids), batch_size):
            batch_ids = ids[offset:offset + batch_size]
            digest = hashlib.sha256(",".join(batch_ids).encode("utf-8")).hexdigest()
            checkpoint = batch_dir / (digest + ".json")
            if checkpoint.is_file():
                saved_batch = json.loads(checkpoint.read_text(encoding="utf-8"))
                papers = saved_batch["papers"]
                if saved_batch.get("ids") != batch_ids or [p["id"] for p in papers] != batch_ids:
                    raise ValueError("Discovery checkpoint does not match assigned IDs")
                elapsed = float(saved_batch["seconds"])
            else:
                failed_before = set(failed_ids)
                started = time.perf_counter()
                papers = fetch(batch_ids)
                elapsed = time.perf_counter() - started
                queried = True
                if [p["id"] for p in papers] != batch_ids:
                    raise ValueError("Discovery returned an incomplete or reordered batch")
                if set(failed_ids) - failed_before or any(p.get("status") == "error" for p in papers):
                    raise RuntimeError("Discovery batch failed after retries; pause and resume later")
                for paper in papers:
                    save(paper)
                write_json(checkpoint, {"ids": batch_ids, "papers": papers, "seconds": elapsed})

            records.extend(papers)
            seconds += elapsed
            if bar is not None:
                bar.update(len(batch_ids))
                bar.set_postfix_str(f"{sum(p.get('versions', 0) > 0 for p in records):,} papers, "
                                    f"{sum(p.get('versions', 0) for p in records):,} versions")
            else:
                print("[discovery] {:,}/{:,} IDs checkpointed ({:.1f}%)".format(
                    len(records), len(ids), 100 * len(records) / len(ids)), flush=True)
    finally:
        if bar is not None:
            bar.close()
    return records, seconds, queried


MANIFEST = STATE_DIR / "discovery_manifest.json"     # every assigned ID with its version count
DISCOVERY_BATCHES = STATE_DIR / "discovery_batches"  # one checkpoint file per finished batch


def discover_all(ids=None, manifest_path=None, batch_dir=None, fetch=None, save=None):
    """Discover every assigned ID once and keep the result in MANIFEST.

    Returns (papers, queried, discovery_seconds); queried is False when nothing had to be asked.
    """
    ids = all_assigned_ids() if ids is None else list(ids)
    manifest_path = Path(MANIFEST if manifest_path is None else manifest_path)
    batch_dir = DISCOVERY_BATCHES if batch_dir is None else batch_dir
    saved = read_json(manifest_path)
    papers = (saved["papers"] if isinstance(saved, dict) else saved) or []
    seconds = saved.get("discovery_seconds", 0.0) if isinstance(saved, dict) else 0.0
    by_id = {p["id"]: p for p in papers}
    if set(by_id) - set(ids):
        raise ValueError("Saved manifest contains IDs outside the assigned ranges")
    missing = [i for i in ids if i not in by_id
               or (by_id[i].get("status") == "error" and by_id[i].get("versions") == 0)]
    print(f"[discovery] manifest has {len(by_id):,}/{len(ids):,} IDs; {len(missing):,} to query")

    queried = False
    if missing:
        updates, elapsed, queried = resolve_discovery_batches(
            missing, fetch or fetch_metadata, save or save_metadata, batch_dir,
            DISCOVERY_STATS["failed_ids"], batch_size=ARXIV_BATCH_SIZE)
        by_id.update({p["id"]: p for p in updates})
        seconds += elapsed
        papers = [by_id[i] for i in ids]
        manifest_path.parent.mkdir(parents=True, exist_ok=True)
        write_json(manifest_path, {"papers": papers, "discovery_seconds": seconds})
    return papers, queried, seconds

## 6. Quick test

The first cell checks the discovery checkpoints offline (fake API, temporary folder); the second
queries the live API. Both run only when this notebook is opened on its own (`RUN_TEST = True`).
The live sample covers a
paper with many versions (`1706.03762`, 7 versions), an old-style ID (`hep-th/9711200`, expected
`partial`, see above), a paper with a journal reference, and an ID that does not exist
(`2310.99999`, expected `not_found`).

In [ ]:
if RUN_TEST:
    # Offline checks of the batch checkpoints and the manifest: fake API, temporary folder
    import tempfile

    def _fake_fetch(calls, failed=None, fail_id=None):
        def fetch(batch):
            calls.append(list(batch))
            if fail_id in batch:
                failed.extend(batch)
                return [{"id": i, "status": "error", "versions": 0} for i in batch]
            return [{"id": i, "status": "ok", "versions": 1} for i in batch]
        return fetch

    _ids = ["2603.20580", "2603.20581", "2603.20582", "2603.20583"]
    with tempfile.TemporaryDirectory() as _tmp:
        _tmp = Path(_tmp)
        _failed, _saved, _calls = [], [], []
        try:
            resolve_discovery_batches(_ids, _fake_fetch(_calls, _failed, _ids[2]), _saved.append,
                                      _tmp / "batches", _failed, batch_size=2)
        except RuntimeError:
            pass
        else:
            raise AssertionError("a batch that keeps failing must stop discovery")
        assert _calls == [_ids[:2], _ids[2:]] and len(list((_tmp / "batches").glob("*.json"))) == 1
        _failed.clear()
        _records, _, _queried = resolve_discovery_batches(_ids, _fake_fetch(_calls), _saved.append,
                                                          _tmp / "batches", _failed, batch_size=2)
        assert _calls == [_ids[:2], _ids[2:], _ids[2:]]   # the saved batch is not queried again
        assert [r["id"] for r in _records] == _ids and [r["id"] for r in _saved] == _ids and _queried

        _calls = []
        _papers, _queried, _ = discover_all(_ids, _tmp / "manifest.json", _tmp / "b2",
                                            _fake_fetch(_calls), lambda paper: None)
        assert _queried and [p["id"] for p in _papers] == _ids
        assert read_json(_tmp / "manifest.json")["papers"] == _papers
        _papers, _queried, _ = discover_all(_ids, _tmp / "manifest.json", _tmp / "b2",
                                            _fake_fetch(_calls), lambda paper: None)
        assert not _queried and len(_calls) == 1          # a complete manifest needs no API call
    print("Offline discovery checkpoint checks passed")

In [ ]:
if RUN_TEST:
    # Offline checks of the ID helpers
    assert list_ids("2310.00098", "2310.00101") == ["2310.00098", "2310.00099", "2310.00100", "2310.00101"]
    assert list_ids("1401.0001", "1401.0002") == ["1401.0001", "1401.0002"]
    assert list_ids("hep-th/9901001", "hep-th/9901002") == ["hep-th/9901001", "hep-th/9901002"]
    assert split_version("2310.12345v2") == ("2310.12345", 2)
    assert to_folder_id("hep-th/9901001v2") == "hep-th_9901001"
    assert [len(list_ids(s, e)) for s, e in ID_RANGES] == [5000, 5000, 5000, 5000]
    assert len(all_assigned_ids()) == 20000

    TEST_IDS = [
        "1706.03762", "hep-th/9711200", "1512.03385", "1810.04805", "2005.14165",
        "1409.0473", "2106.09685", "2201.11903", "2310.06825", "2310.99999",
    ]
    test_papers = fetch_metadata(TEST_IDS)
    for p in test_papers:
        save_metadata(p)
        print(f"{p['id']:<16} {p['status']:<10} versions={p['versions']}  "
              f"revised={len(p.get('revised', []))}  venue={p.get('venue')}")
    print(DISCOVERY_STATS)